# Filtrado colaborativo basado en usuarios

## Importación de librerías

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

## Carga de datos

In [8]:
DATA_DIR = Path("../data/raw/ml-latest-small")

ratings_path = DATA_DIR / "ratings.csv"
movies_path = DATA_DIR / "movies.csv"

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

In [5]:
print("Ratings:", ratings.shape)
print("Movies:", movies.shape)

ratings.head()

Ratings: (100836, 4)
Movies: (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Construcción de la matriz usuario–película

In [6]:
matriz_usuario_pelicula = ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

matriz_usuario_pelicula.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Preparación de la matriz para calcular similitudes

In [9]:
matriz_similitud = matriz_usuario_pelicula.fillna(0)

matriz_similitud.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Cálculo de la similitud entre usuarios

In [10]:
similitud_usuarios = cosine_similarity(matriz_similitud)

In [11]:
similitud_usuarios = pd.DataFrame(
    similitud_usuarios,
    index=matriz_usuario_pelicula.index,
    columns=matriz_usuario_pelicula.index
)

similitud_usuarios.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [12]:
print(similitud_usuarios.shape)

similitud_usuarios.iloc[:5, :5]

(610, 610)


userId,1,2,3,4,5
userId,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080
2,0.027283,1.000000,0.000000,0.003726,0.016614
3,0.059720,0.000000,1.000000,0.002251,0.005020
4,0.194395,0.003726,0.002251,1.000000,0.128659
5,0.129080,0.016614,0.005020,0.128659,1.000000


## Selección de los vecinos más similares

In [13]:
usuario_objetivo = 1

In [14]:
similitudes = similitud_usuarios.loc[usuario_objetivo]

similitudes.head()

userId
1    1.000000
2    0.027283
3    0.059720
4    0.194395
5    0.129080
Name: 1, dtype: float64

In [15]:
similitudes.sort_values(ascending=False).head(10)

userId
1      1.000000
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
Name: 1, dtype: float64

In [16]:
similitudes = similitudes.drop(usuario_objetivo)

similitudes.sort_values(ascending=False).head(10)

userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
45     0.327922
Name: 1, dtype: float64

In [17]:
k = 10

vecinos = similitudes.sort_values(ascending=False).head(k)

vecinos

userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
45     0.327922
Name: 1, dtype: float64

## Películas no vistas por el usuario

In [18]:
peliculas_usuario = matriz_usuario_pelicula.loc[usuario_objetivo]

peliculas_no_vistas = peliculas_usuario[peliculas_usuario.isna()].index

print(f"Películas no vistas: {len(peliculas_no_vistas)}")

Películas no vistas: 9492


## Predicción de valoraciones a partir de los vecinos

In [27]:
pelicula_objetivo = peliculas_no_vistas[0]

pelicula_objetivo

np.int64(2)

In [28]:
valoraciones_vecinos = matriz_usuario_pelicula.loc[vecinos.index, pelicula_objetivo]

valoraciones_vecinos

userId
266    NaN
313    NaN
368    NaN
57     NaN
91     3.0
469    NaN
39     NaN
288    2.0
452    NaN
45     NaN
Name: 2, dtype: float64

In [29]:
vecinos_que_la_han_visto = valoraciones_vecinos.dropna()

vecinos_que_la_han_visto

userId
91     3.0
288    2.0
Name: 2, dtype: float64

In [30]:
similitudes_validas = vecinos.loc[vecinos_que_la_han_visto.index]

similitudes_validas

userId
91     0.334727
288    0.329700
Name: 1, dtype: float64

In [31]:
prediccion = (
    vecinos_que_la_han_visto * similitudes_validas
).sum() / similitudes_validas.sum()

print(f"Predicción para la película {pelicula_objetivo}: {prediccion:.2f}")

Predicción para la película 2: 2.50


## Función para predecir la valoración de una película

In [32]:
def predecir_valoracion(usuario_id, pelicula_id, vecinos):
    valoraciones_vecinos = matriz_usuario_pelicula.loc[
        vecinos.index,
        pelicula_id
    ]

    vecinos_que_la_han_visto = valoraciones_vecinos.dropna()

    if vecinos_que_la_han_visto.empty:
        return np.nan

    similitudes_validas = vecinos.loc[
        vecinos_que_la_han_visto.index
    ]

    prediccion = (
        vecinos_que_la_han_visto * similitudes_validas
    ).sum() / similitudes_validas.sum()

    return prediccion

In [33]:
prediccion_prueba = predecir_valoracion(
    usuario_objetivo,
    pelicula_objetivo,
    vecinos
)

print(f"Predicción: {prediccion_prueba:.2f}")

Predicción: 2.50


## Predicción de todas las películas no vistas

In [34]:
predicciones = {}

for pelicula_id in peliculas_no_vistas:
    prediccion = predecir_valoracion(
        usuario_objetivo,
        pelicula_id,
        vecinos
    )

    if not np.isnan(prediccion):
        predicciones[pelicula_id] = prediccion

In [35]:
predicciones = pd.Series(predicciones)

predicciones_ordenadas = predicciones.sort_values(
    ascending=False
)

predicciones_ordenadas.head(10)

1243     5.0
3022     5.0
4718     5.0
1235     5.0
4734     5.0
6460     5.0
3730     5.0
3813     5.0
54272    5.0
1449     5.0
dtype: float64

## Mostrar los títulos de las películas recomendadas

In [36]:
recomendaciones = pd.DataFrame({
    "movieId": predicciones_ordenadas.head(10).index,
    "prediccion": predicciones_ordenadas.head(10).values
})

recomendaciones

,movieId,prediccion
0,1243,5.0
1,3022,5.0
2,4718,5.0
3,1235,5.0
4,4734,5.0
5,6460,5.0
6,3730,5.0
7,3813,5.0
8,54272,5.0
9,1449,5.0


In [37]:
recomendaciones = recomendaciones.merge(
    movies,
    on="movieId"
)

recomendaciones

,movieId,prediccion,title,genres
0,1243,5.0,Rosencrantz and Guildenstern Are Dead (1990),Comedy|Drama
1,3022,5.0,"General, The (1926)",Comedy|War
2,4718,5.0,American Pie 2 (2001),Comedy
3,1235,5.0,Harold and Maude (1971),Comedy|Drama|Romance
4,4734,5.0,Jay and Silent Bob Strike Back (2001),Adventure|Comedy
5,6460,5.0,"Trial, The (Procès, Le) (1962)",Drama
6,3730,5.0,"Conversation, The (1974)",Drama|Mystery
7,3813,5.0,Interiors (1978),Drama
8,54272,5.0,"Simpsons Movie, The (2007)",Animation|Comedy
9,1449,5.0,Waiting for Guffman (1996),Comedy


In [38]:
recomendaciones[
    ["title", "prediccion", "genres"]
]

,title,prediccion,genres
0,Rosencrantz and Guildenstern Are Dead (1990),5.0,Comedy|Drama
1,"General, The (1926)",5.0,Comedy|War
2,American Pie 2 (2001),5.0,Comedy
3,Harold and Maude (1971),5.0,Comedy|Drama|Romance
4,Jay and Silent Bob Strike Back (2001),5.0,Adventure|Comedy
5,"Trial, The (Procès, Le) (1962)",5.0,Drama
6,"Conversation, The (1974)",5.0,Drama|Mystery
7,Interiors (1978),5.0,Drama
8,"Simpsons Movie, The (2007)",5.0,Animation|Comedy
9,Waiting for Guffman (1996),5.0,Comedy


## Normalización de las valoraciones por usuario

In [39]:
media_por_usuario = matriz_usuario_pelicula.mean(axis=1)

matriz_normalizada = matriz_usuario_pelicula.sub(
    media_por_usuario,
    axis=0
)

matriz_normalizada.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.366379,NaN,-0.366379,NaN,NaN,-0.366379,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.363636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cálculo de similitudes con valoraciones normalizadas

In [41]:
matriz_normalizada_similitud = matriz_normalizada.fillna(0)

In [42]:
similitud_usuarios_normalizada = cosine_similarity(
    matriz_normalizada_similitud
)

In [43]:
similitud_usuarios_normalizada = pd.DataFrame(
    similitud_usuarios_normalizada,
    index=matriz_usuario_pelicula.index,
    columns=matriz_usuario_pelicula.index
)

similitud_usuarios_normalizada.iloc[:5, :5]

userId,1,2,3,4,5
userId,,,,,
1,1.000000,0.001265,0.000553,0.048419,0.021847
2,0.001265,1.000000,0.000000,-0.017164,0.021796
3,0.000553,0.000000,1.000000,-0.011260,-0.031539
4,0.048419,-0.017164,-0.011260,1.000000,-0.029620
5,0.021847,0.021796,-0.031539,-0.029620,1.000000


## Selección de vecinos con similitud normalizada

In [44]:
similitudes_normalizadas = similitud_usuarios_normalizada.loc[usuario_objetivo]

similitudes_normalizadas = similitudes_normalizadas.drop(usuario_objetivo)

vecinos_normalizados = (
    similitudes_normalizadas
    .sort_values(ascending=False)
    .head(k)
)

vecinos_normalizados

userId
301    0.124799
597    0.102631
414    0.101348
477    0.099217
57     0.099070
369    0.098295
206    0.096852
535    0.096493
590    0.095191
418    0.094153
Name: 1, dtype: float64

In [45]:
print("Vecinos originales:")
display(vecinos)

print("Vecinos normalizados:")
display(vecinos_normalizados)

Vecinos originales:


userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
45     0.327922
Name: 1, dtype: float64

Vecinos normalizados:


userId
301    0.124799
597    0.102631
414    0.101348
477    0.099217
57     0.099070
369    0.098295
206    0.096852
535    0.096493
590    0.095191
418    0.094153
Name: 1, dtype: float64

## Predicción normalizada de valoraciones

In [65]:
def predecir_valoracion_normalizada(
    usuario_id,
    pelicula_id,
    vecinos_normalizados,
    min_vecinos=3
):
    # Valoraciones normalizadas de los vecinos para esa película
    desviaciones_vecinos = matriz_normalizada.loc[
        vecinos_normalizados.index,
        pelicula_id
    ]

    # Nos quedamos solo con quienes realmente la han valorado
    desviaciones_validas = desviaciones_vecinos.dropna()

    # Exigimos un mínimo de vecinos
    if len(desviaciones_validas) < min_vecinos:
        return np.nan

    # Recuperamos únicamente las similitudes de esos vecinos
    similitudes_validas = vecinos_normalizados.loc[
        desviaciones_validas.index
    ]

    # Predicción de la desviación respecto a la media
    desviacion_predicha = (
        desviaciones_validas * similitudes_validas
    ).sum() / similitudes_validas.abs().sum()

    # Volvemos a la escala original del usuario objetivo
    prediccion = media_por_usuario.loc[usuario_id] + desviacion_predicha

    # Limitamos la predicción al rango válido
    prediccion = np.clip(
        prediccion,
        0.5,
        5.0
    )

    return prediccion

In [66]:
prediccion_normalizada_prueba = predecir_valoracion_normalizada(
    usuario_objetivo,
    pelicula_objetivo,
    vecinos_normalizados
)

print("Media del usuario:", media_por_usuario.loc[usuario_objetivo])
print("Predicción normalizada:", prediccion_normalizada_prueba)

Media del usuario: 4.366379310344827
Predicción normalizada: 4.044970934385208


## Generación de recomendaciones con el modelo mejorado

In [67]:
peliculas_no_vistas = matriz_usuario_pelicula.loc[
    usuario_objetivo
].isna()

peliculas_no_vistas = peliculas_no_vistas[
    peliculas_no_vistas
].index

len(peliculas_no_vistas)

9492

In [68]:
predicciones_mejoradas = {}

for pelicula_id in peliculas_no_vistas:

    prediccion = predecir_valoracion_normalizada(
        usuario_objetivo,
        pelicula_id,
        vecinos_normalizados,
        min_vecinos=3
    )

    if not np.isnan(prediccion):
        predicciones_mejoradas[pelicula_id] = prediccion

In [69]:
predicciones_mejoradas = pd.Series(
    predicciones_mejoradas
)

predicciones_mejoradas = predicciones_mejoradas.sort_values(
    ascending=False
)

predicciones_mejoradas.head(10)

58559    5.0
54503    5.0
55820    5.0
46578    5.0
348      5.0
48516    5.0
48780    5.0
318      5.0
30707    5.0
30749    5.0
dtype: float64

In [70]:
recomendaciones_mejoradas = pd.DataFrame({

    "movieId": predicciones_mejoradas.head(10).index,

    "prediccion": predicciones_mejoradas.head(10).values

})

In [71]:
recomendaciones_mejoradas = recomendaciones_mejoradas.merge(

    movies,

    on="movieId"

)

recomendaciones_mejoradas[
    ["title", "prediccion", "genres"]
]

,title,prediccion,genres
0,"Dark Knight, The (2008)",5.0,Action|Crime|Drama|IMAX
1,Superbad (2007),5.0,Comedy
2,No Country for Old Men (2007),5.0,Crime|Drama
3,Little Miss Sunshine (2006),5.0,Adventure|Comedy|Drama
4,Bullets Over Broadway (1994),5.0,Comedy
5,"Departed, The (2006)",5.0,Crime|Drama|Thriller
6,"Prestige, The (2006)",5.0,Drama|Mystery|Sci-Fi|Thriller
7,"Shawshank Redemption, The (1994)",5.0,Crime|Drama
8,Million Dollar Baby (2004),5.0,Drama
9,Hotel Rwanda (2004),5.0,Drama|War


In [74]:
predicciones_mejoradas.describe()

count    432.000000
mean       4.392140
std        0.643608
min        2.011747
25%        3.995665
50%        4.630286
75%        4.987316
max        5.000000
dtype: float64

In [75]:
predicciones_mejoradas.value_counts().head(10)

5.000000    105
4.709502      7
4.787602      4
4.985774      3
4.877237      3
4.864904      3
4.716309      3
4.544971      3
4.044971      3
4.038165      3
Name: count, dtype: int64